# 03 Aggregations

Use Spark SQL aggregations and connect GROUP BY to shuffle reasoning.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("module-02-spark-sql").getOrCreate()
base = "../../datasets/module_02"

In [ ]:
municipalities = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/municipalities.csv")
accessibility = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/accessibility_scores.csv")
poi = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/poi_counts.csv")
property_values = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/property_values.csv")
municipalities.printSchema()
municipalities.show(5, truncate=False)

In [ ]:
municipalities.createOrReplaceTempView("municipalities")
accessibility.createOrReplaceTempView("accessibility_scores")
poi.createOrReplaceTempView("poi_counts")
property_values.createOrReplaceTempView("property_values")

In [ ]:
spark.sql("""
SELECT m.canton,
       COUNT(*) AS municipality_count,
       ROUND(AVG(a.accessibility_score), 1) AS avg_accessibility,
       SUM(m.population) AS total_population
FROM municipalities m
JOIN accessibility_scores a USING (municipality_id)
GROUP BY m.canton
ORDER BY avg_accessibility DESC
""").show(truncate=False)

In [ ]:
spark.sql("""
EXPLAIN FORMATTED
SELECT m.canton, AVG(a.accessibility_score) AS avg_accessibility
FROM municipalities m
JOIN accessibility_scores a USING (municipality_id)
GROUP BY m.canton
""").show(truncate=False)